# Perbandingan 3 Optimizer pada Concrete Compressive Strength

In [1]:
import os
import glob
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fmin_l_bfgs_b
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
HIDDEN_DIM = 6
REG = 1e-4

COMMON_MAX_ITER = 1000
GD_LR = 0.02
GD_GRAD_CLIP = 5.0
GN_DAMPING = 1e-2

USE_EARLY_STOPPING = False

TEST_SIZE = 0.15
VAL_SIZE_FROM_TRAIN = 0.20
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = WORKING_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42) -> np.random.Generator:
    np.random.seed(seed)
    return np.random.default_rng(seed)


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true, y_pred, eps: float = 1e-8) -> float:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def ensure_numeric_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(axis=0).reset_index(drop=True)
    return out


def find_dataset_file() -> Path:
    candidates = []
    search_patterns = [
        "/kaggle/input/**/*.csv",
        "/kaggle/input/**/*.xls",
        "/kaggle/input/**/*.xlsx",
        "./*.csv",
        "./*.xls",
        "./*.xlsx",
    ]
    for pattern in search_patterns:
        candidates.extend(glob.glob(pattern, recursive=True))

    if not candidates:
        raise FileNotFoundError(
            "Dataset tidak ditemukan. Pastikan dataset Kaggle sudah di-attach ke notebook."
        )

    concrete_like = []
    keywords = ["concrete", "compressive", "strength"]
    for c in candidates:
        lower = c.lower()
        if any(k in lower for k in keywords):
            concrete_like.append(c)

    chosen = concrete_like[0] if concrete_like else candidates[0]
    return Path(chosen)


def load_concrete_data() -> tuple[pd.DataFrame, str, Path]:
    path = find_dataset_file()
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xls", ".xlsx"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Format file tidak didukung: {path}")

    df.columns = [str(c).strip() for c in df.columns]
    df = ensure_numeric_dataframe(df)

    target_candidates = [
        c for c in df.columns
        if "compressive strength" in c.lower()
        or "concrete compressive strength" in c.lower()
        or "strength" == c.lower().strip()
    ]
    target_col = target_candidates[0] if target_candidates else df.columns[-1]

    return df, target_col, path


def unpack_params(theta: np.ndarray, d: int, h: int):
    idx = 0
    W1 = theta[idx:idx + d * h].reshape(d, h)
    idx += d * h
    b1 = theta[idx:idx + h]
    idx += h
    W2 = theta[idx:idx + h].reshape(h, 1)
    idx += h
    b2 = theta[idx]
    return W1, b1, W2, b2


def forward(theta: np.ndarray, X: np.ndarray, d: int, h: int):
    W1, b1, W2, b2 = unpack_params(theta, d, h)
    Z = X @ W1 + b1
    H = np.tanh(Z)
    y_hat = (H @ W2 + b2).ravel()
    return y_hat, H, Z


def predict(theta: np.ndarray, X: np.ndarray, d: int, h: int) -> np.ndarray:
    y_hat, _, _ = forward(theta, X, d, h)
    return y_hat


def residual_and_jacobian(theta: np.ndarray, X: np.ndarray, y: np.ndarray, d: int, h: int):
    W1, b1, W2, b2 = unpack_params(theta, d, h)
    Z = X @ W1 + b1
    H = np.tanh(Z)
    y_hat = (H @ W2 + b2).ravel()
    r = y_hat - y
    sech2 = 1.0 - H ** 2

    J = np.zeros((X.shape[0], theta.size), dtype=np.float64)
    idx = 0

    for feature_idx in range(d):
        for hidden_idx in range(h):
            J[:, idx] = W2[hidden_idx, 0] * sech2[:, hidden_idx] * X[:, feature_idx]
            idx += 1

    for hidden_idx in range(h):
        J[:, idx] = W2[hidden_idx, 0] * sech2[:, hidden_idx]
        idx += 1

    for hidden_idx in range(h):
        J[:, idx] = H[:, hidden_idx]
        idx += 1

    J[:, idx] = 1.0
    return r, J


def loss_grad(theta: np.ndarray, X: np.ndarray, y: np.ndarray, d: int, h: int, reg: float = 0.0):
    n = X.shape[0]
    W1, b1, W2, b2 = unpack_params(theta, d, h)
    Z = X @ W1 + b1
    H = np.tanh(Z)
    y_hat = (H @ W2 + b2).ravel()
    r = y_hat - y

    loss = float(np.mean(r ** 2) + reg * np.sum(theta ** 2))

    # Backprop gradient for full-batch optimization
    sech2 = 1.0 - H ** 2
    dy = (2.0 / n) * r[:, None]
    grad_W2 = H.T @ dy + 2.0 * reg * W2
    grad_b2 = float(dy.sum() + 2.0 * reg * b2)
    dH = dy @ W2.T
    dZ = dH * sech2
    grad_W1 = X.T @ dZ + 2.0 * reg * W1
    grad_b1 = dZ.sum(axis=0) + 2.0 * reg * b1

    grad = np.concatenate([
        grad_W1.ravel(),
        grad_b1.ravel(),
        grad_W2.ravel(),
        np.array([grad_b2], dtype=np.float64),
    ])
    return loss, grad, r


def train_gd(
    X_train,
    y_train,
    X_val,
    y_val,
    d,
    h,
    theta0,
    lr=0.02,
    max_iter=100,
    reg=1e-4,
    grad_clip=5.0,
    tol=1e-8,
    use_early_stopping=False,
):
    theta = theta0.copy()
    history = []
    best_theta = theta.copy()
    best_val = np.inf
    start = time.time()

    for iteration in range(1, max_iter + 1):
        train_loss, grad, _ = loss_grad(theta, X_train, y_train, d, h, reg)
        grad_norm = float(np.linalg.norm(grad))

        if grad_clip is not None and grad_norm > grad_clip:
            grad = grad * (grad_clip / (grad_norm + 1e-12))
            grad_norm = float(np.linalg.norm(grad))

        theta = theta - lr * grad
        val_loss, _, _ = loss_grad(theta, X_val, y_val, d, h, reg)

        history.append({
            "optimizer": "Gradient Descent",
            "iteration": iteration,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "step_norm": grad_norm,
            "elapsed_sec": time.time() - start,
        })

        if val_loss < best_val:
            best_val = val_loss
            best_theta = theta.copy()

        if use_early_stopping and grad_norm < tol:
            break

    if not use_early_stopping and history:
        best_theta = theta.copy()

    return best_theta, pd.DataFrame(history)


def train_gauss_newton(
    X_train,
    y_train,
    X_val,
    y_val,
    d,
    h,
    theta0,
    max_iter=100,
    reg=1e-4,
    damping=1e-2,
    tol=1e-8,
    use_early_stopping=False,
):
    theta = theta0.copy()
    history = []
    best_theta = theta.copy()
    best_val = np.inf
    identity = np.eye(theta.size)
    start = time.time()

    for iteration in range(1, max_iter + 1):
        r, J = residual_and_jacobian(theta, X_train, y_train, d, h)
        # Damped Gauss-Newton / second-order approximation
        A = (J.T @ J) / len(X_train) + (damping + reg) * identity
        g = (J.T @ r) / len(X_train) + reg * theta
        step = np.linalg.solve(A, g)
        theta = theta - step

        train_loss, _, _ = loss_grad(theta, X_train, y_train, d, h, reg)
        val_loss, _, _ = loss_grad(theta, X_val, y_val, d, h, reg)
        step_norm = float(np.linalg.norm(step))

        history.append({
            "optimizer": "Gauss-Newton",
            "iteration": iteration,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "step_norm": step_norm,
            "elapsed_sec": time.time() - start,
        })

        if val_loss < best_val:
            best_val = val_loss
            best_theta = theta.copy()

        if use_early_stopping and step_norm < tol:
            break

    if not use_early_stopping and history:
        best_theta = theta.copy()

    return best_theta, pd.DataFrame(history)


def train_lbfgs(
    X_train,
    y_train,
    X_val,
    y_val,
    d,
    h,
    theta0,
    max_iter=100,
    reg=1e-4,
):
    history_with_params = []
    start = time.time()

    def objective(theta):
        return loss_grad(theta, X_train, y_train, d, h, reg)[:2]

    def callback(xk):
        train_loss, _, _ = loss_grad(xk, X_train, y_train, d, h, reg)
        val_loss, _, _ = loss_grad(xk, X_val, y_val, d, h, reg)
        history_with_params.append({
            "iteration": len(history_with_params) + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "elapsed_sec": time.time() - start,
            "theta": xk.copy(),
        })

    result_theta, result_loss, info = fmin_l_bfgs_b(
        func=lambda th: objective(th),
        x0=theta0.copy(),
        maxiter=max_iter,
        pgtol=0.0,
        factr=0.0,
        callback=callback,
    )

    if not history_with_params:
        x = result_theta.copy()
        train_loss, _, _ = loss_grad(x, X_train, y_train, d, h, reg)
        val_loss, _, _ = loss_grad(x, X_val, y_val, d, h, reg)
        history_with_params.append({
            "iteration": 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "elapsed_sec": time.time() - start,
            "theta": x.copy(),
        })

    history = []
    prev_theta = None
    for row in history_with_params:
        step_norm = np.nan if prev_theta is None else float(np.linalg.norm(row["theta"] - prev_theta))
        history.append({
            "optimizer": "L-BFGS",
            "iteration": row["iteration"],
            "train_loss": row["train_loss"],
            "val_loss": row["val_loss"],
            "step_norm": step_norm,
            "elapsed_sec": row["elapsed_sec"],
        })
        prev_theta = row["theta"]

    return result_theta, pd.DataFrame(history), info


def inverse_transform_y(y_scaled, y_scaler: StandardScaler):
    return y_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).ravel()


def evaluate_model(name, theta, X_train, y_train, X_val, y_val, X_test, y_test, d, h, y_scaler):
    pred_train_scaled = predict(theta, X_train, d, h)
    pred_val_scaled = predict(theta, X_val, d, h)
    pred_test_scaled = predict(theta, X_test, d, h)

    y_train_orig = inverse_transform_y(y_train, y_scaler)
    y_val_orig = inverse_transform_y(y_val, y_scaler)
    y_test_orig = inverse_transform_y(y_test, y_scaler)
    pred_train_orig = inverse_transform_y(pred_train_scaled, y_scaler)
    pred_val_orig = inverse_transform_y(pred_val_scaled, y_scaler)
    pred_test_orig = inverse_transform_y(pred_test_scaled, y_scaler)

    metrics = {
        "optimizer": name,
        "train_rmse": rmse(y_train_orig, pred_train_orig),
        "val_rmse": rmse(y_val_orig, pred_val_orig),
        "test_rmse": rmse(y_test_orig, pred_test_orig),
        "train_mae": float(mean_absolute_error(y_train_orig, pred_train_orig)),
        "val_mae": float(mean_absolute_error(y_val_orig, pred_val_orig)),
        "test_mae": float(mean_absolute_error(y_test_orig, pred_test_orig)),
        "train_r2": float(r2_score(y_train_orig, pred_train_orig)),
        "val_r2": float(r2_score(y_val_orig, pred_val_orig)),
        "test_r2": float(r2_score(y_test_orig, pred_test_orig)),
        "test_mape": mape(y_test_orig, pred_test_orig),
    }

    predictions = pd.DataFrame({
        "split": ["train"] * len(y_train_orig) + ["val"] * len(y_val_orig) + ["test"] * len(y_test_orig),
        "optimizer": name,
        "y_true": np.concatenate([y_train_orig, y_val_orig, y_test_orig]),
        "y_pred": np.concatenate([pred_train_orig, pred_val_orig, pred_test_orig]),
        "residual": np.concatenate([
            y_train_orig - pred_train_orig,
            y_val_orig - pred_val_orig,
            y_test_orig - pred_test_orig,
        ]),
    })

    return metrics, predictions


def save_plot_loss_curves(history_df: pd.DataFrame):
    plt.figure(figsize=(10, 6))
    for name in history_df["optimizer"].unique():
        part = history_df[history_df["optimizer"] == name]
        plt.plot(part["iteration"], part["train_loss"], label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Train Loss (scaled MSE)")
    plt.title("Training Loss Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "01_training_loss_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 6))
    for name in history_df["optimizer"].unique():
        part = history_df[history_df["optimizer"] == name]
        plt.plot(part["iteration"], part["val_loss"], label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Validation Loss (scaled MSE)")
    plt.title("Validation Loss Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "02_validation_loss_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 6))
    for name in history_df["optimizer"].unique():
        part = history_df[history_df["optimizer"] == name]
        plt.plot(part["iteration"], part["step_norm"], label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Step / Gradient Norm")
    plt.title("Step Norm Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "03_step_norm_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(10, 6))
    for name in history_df["optimizer"].unique():
        part = history_df[history_df["optimizer"] == name]
        plt.plot(part["elapsed_sec"], part["val_loss"], label=name)
    plt.xlabel("Elapsed Time (seconds)")
    plt.ylabel("Validation Loss (scaled MSE)")
    plt.title("Validation Loss vs Time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "04_validation_loss_vs_time.png", dpi=200)
    plt.close()


def save_plot_metrics(summary_df: pd.DataFrame):
    bar_df = summary_df[["optimizer", "test_rmse", "test_mae", "test_r2", "fit_time_sec", "n_iterations"]].copy()

    plt.figure(figsize=(9, 6))
    plt.bar(bar_df["optimizer"], bar_df["test_rmse"])
    plt.ylabel("Test RMSE")
    plt.title("Comparison of Test RMSE")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "05_test_rmse_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 6))
    plt.bar(bar_df["optimizer"], bar_df["test_mae"])
    plt.ylabel("Test MAE")
    plt.title("Comparison of Test MAE")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "06_test_mae_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 6))
    plt.bar(bar_df["optimizer"], bar_df["test_r2"])
    plt.ylabel("Test R2")
    plt.title("Comparison of Test R2")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "07_test_r2_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 6))
    plt.bar(bar_df["optimizer"], bar_df["fit_time_sec"])
    plt.ylabel("Fit Time (seconds)")
    plt.title("Comparison of Fit Time")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "08_fit_time_comparison.png", dpi=200)
    plt.close()

    plt.figure(figsize=(9, 6))
    plt.bar(bar_df["optimizer"], bar_df["n_iterations"])
    plt.ylabel("Number of Iterations")
    plt.title("Comparison of Iterations")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "09_iteration_comparison.png", dpi=200)
    plt.close()


def save_prediction_and_residual_plots(all_predictions: pd.DataFrame):
    test_df = all_predictions[all_predictions["split"] == "test"].copy()

    for optimizer_name in test_df["optimizer"].unique():
        part = test_df[test_df["optimizer"] == optimizer_name]

        plt.figure(figsize=(7, 7))
        plt.scatter(part["y_true"], part["y_pred"], alpha=0.7)
        low = min(part["y_true"].min(), part["y_pred"].min())
        high = max(part["y_true"].max(), part["y_pred"].max())
        plt.plot([low, high], [low, high], linestyle="--")
        safe_name = optimizer_name.lower().replace("-", "").replace(" ", "_")
        plt.xlabel("Actual Strength")
        plt.ylabel("Predicted Strength")
        plt.title(f"Actual vs Predicted - {optimizer_name}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"10_actual_vs_predicted_{safe_name}.png", dpi=200)
        plt.close()

        plt.figure(figsize=(8, 6))
        plt.scatter(part["y_pred"], part["residual"], alpha=0.7)
        plt.axhline(0.0, linestyle="--")
        plt.xlabel("Predicted Strength")
        plt.ylabel("Residual (Actual - Predicted)")
        plt.title(f"Residual Plot - {optimizer_name}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"11_residual_plot_{safe_name}.png", dpi=200)
        plt.close()

        plt.figure(figsize=(8, 6))
        plt.hist(part["residual"], bins=25)
        plt.xlabel("Residual")
        plt.ylabel("Frequency")
        plt.title(f"Residual Histogram - {optimizer_name}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"12_residual_histogram_{safe_name}.png", dpi=200)
        plt.close()


def create_section4_markdown(summary_df: pd.DataFrame, history_df: pd.DataFrame, dataset_meta: dict):
    best_rmse_row = summary_df.sort_values("test_rmse", ascending=True).iloc[0]
    best_r2_row = summary_df.sort_values("test_r2", ascending=False).iloc[0]
    fastest_row = summary_df.sort_values("fit_time_sec", ascending=True).iloc[0]
    fewest_iter_row = summary_df.sort_values("n_iterations", ascending=True).iloc[0]

    lines = []
    lines.append("# 4. Hasil dan Pembahasan\n")
    lines.append("## 4.1 Ringkasan Dataset dan Eksperimen\n")
    lines.append(
        f"Eksperimen menggunakan dataset **Concrete Compressive Strength** dengan "
        f"**{dataset_meta['n_rows']}** observasi dan **{dataset_meta['n_features']}** fitur input. "
        f"Target yang diprediksi adalah **{dataset_meta['target_col']}**. "
        f"Data dibagi menjadi train/validation/test = 60/20/20 dengan standardisasi pada fitur dan target.\n"
    )
    lines.append(
        "Model yang digunakan adalah neural network nonlinear 1 hidden layer dengan aktivasi tanh, "
        "sehingga ketiga optimizer meminimasi objective yang sama, yaitu Mean Squared Error (MSE) "
        "dengan regularisasi L2 ringan. Selain itu, ketiga optimizer menggunakan inisialisasi parameter "
        "awal yang sama, split data yang sama, dan budget iterasi maksimum yang sama (**100 iterasi**). "
        "Formulasi ini membuat perbandingan Gradient Descent, "
        "Gauss-Newton, dan L-BFGS menjadi lebih adil pada objective nonlinear least-squares yang sama.\n"
    )

    lines.append("## 4.2 Perbandingan Kinerja Akhir\n")
    lines.append(summary_df.to_markdown(index=False) + "\n")
    lines.append(
        f"Berdasarkan metrik uji, optimizer dengan **RMSE test terbaik** adalah **{best_rmse_row['optimizer']}** "
        f"dengan nilai **{best_rmse_row['test_rmse']:.4f}**. "
        f"Optimizer dengan **R2 test tertinggi** juga adalah **{best_r2_row['optimizer']}** "
        f"dengan nilai **{best_r2_row['test_r2']:.4f}**.\n"
    )

    lines.append("## 4.3 Analisis Konvergensi\n")
    lines.append(
        f"Dari sisi waktu komputasi, optimizer tercepat adalah **{fastest_row['optimizer']}** "
        f"dengan waktu fitting **{fastest_row['fit_time_sec']:.4f} detik**. "
        f"Dari sisi jumlah iterasi, optimizer yang membutuhkan iterasi paling sedikit adalah "
        f"**{fewest_iter_row['optimizer']}** dengan **{int(fewest_iter_row['n_iterations'])} iterasi**.\n"
    )
    lines.append(
        "Kurva training loss, validation loss, dan step norm menunjukkan karakter konvergensi yang berbeda. "
        "Gradient Descent biasanya mengalami penurunan loss yang stabil tetapi lebih lambat. "
        "Gauss-Newton cenderung mencapai area minimum lebih cepat karena memanfaatkan aproksimasi informasi orde dua melalui matriks Jacobian. "
        "L-BFGS juga memanfaatkan informasi kuasi-Newton sehingga sering menghasilkan kompromi yang baik antara kecepatan konvergensi dan kestabilan numerik.\n"
    )

    lines.append("## 4.4 Analisis Prediksi dan Residual\n")
    lines.append(
        "Plot aktual vs prediksi digunakan untuk melihat seberapa dekat titik-titik prediksi terhadap garis identitas. "
        "Semakin rapat sebaran titik ke garis diagonal, semakin baik kualitas prediksi model. "
        "Residual plot dipakai untuk melihat apakah galat tersebar acak di sekitar nol. "
        "Histogram residual membantu menilai apakah galat terpusat di sekitar nol dan tidak menunjukkan bias sistematis yang kuat.\n"
    )
    lines.append(
        "Apabila residual masih menunjukkan pola tertentu, maka model nonlinear 1 hidden layer dapat ditingkatkan lagi, "
        "misalnya dengan tuning jumlah hidden unit, regularisasi, atau strategi inisialisasi parameter.\n"
    )

    lines.append("## 4.5 Kesimpulan Pembahasan\n")
    lines.append(
        f"Secara umum, hasil eksperimen menunjukkan bahwa **{best_rmse_row['optimizer']}** memberikan performa prediksi terbaik pada data uji, "
        f"sedangkan **{fastest_row['optimizer']}** merupakan optimizer tercepat berdasarkan waktu komputasi. "
        "Dengan demikian, pemilihan optimizer perlu mempertimbangkan dua aspek sekaligus, yaitu kualitas solusi akhir dan efisiensi proses optimasi.\n"
    )

    md_text = "\n".join(lines)
    (WORKING_DIR / "section_4_hasil_dan_pembahasan.md").write_text(md_text, encoding="utf-8")


def main():
    rng = set_seed(RANDOM_STATE)
    print("[INFO] Loading dataset...")
    print(f"[INFO] Fair comparison mode: same initialization, same data split, same objective, same iteration budget = {COMMON_MAX_ITER}")
    df, target_col, dataset_path = load_concrete_data()

    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols].values.astype(np.float64)
    y = df[target_col].values.astype(np.float64)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=VAL_SIZE_FROM_TRAIN,
        random_state=RANDOM_STATE,
    )

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_s = x_scaler.fit_transform(X_train)
    X_val_s = x_scaler.transform(X_val)
    X_test_s = x_scaler.transform(X_test)

    y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
    y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel()
    y_test_s = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

    d = X_train_s.shape[1]
    h = HIDDEN_DIM
    n_params = d * h + h + h + 1
    theta0 = rng.normal(loc=0.0, scale=0.10, size=n_params)

    print("[INFO] Training Gradient Descent...")
    gd_start = time.time()
    gd_theta, gd_history = train_gd(
        X_train_s, y_train_s, X_val_s, y_val_s,
        d=d, h=h, theta0=theta0,
        lr=GD_LR, max_iter=COMMON_MAX_ITER, reg=REG, grad_clip=GD_GRAD_CLIP,
        use_early_stopping=USE_EARLY_STOPPING,
    )
    gd_fit_time = time.time() - gd_start

    print("[INFO] Training Gauss-Newton...")
    gn_start = time.time()
    gn_theta, gn_history = train_gauss_newton(
        X_train_s, y_train_s, X_val_s, y_val_s,
        d=d, h=h, theta0=theta0,
        max_iter=COMMON_MAX_ITER, reg=REG, damping=GN_DAMPING,
        use_early_stopping=USE_EARLY_STOPPING,
    )
    gn_fit_time = time.time() - gn_start

    print("[INFO] Training L-BFGS...")
    lbfgs_start = time.time()
    lbfgs_theta, lbfgs_history, lbfgs_result = train_lbfgs(
        X_train_s, y_train_s, X_val_s, y_val_s,
        d=d, h=h, theta0=theta0,
        max_iter=COMMON_MAX_ITER, reg=REG,
    )
    lbfgs_fit_time = time.time() - lbfgs_start

    history_df = pd.concat([gd_history, gn_history, lbfgs_history], ignore_index=True)

    print("[INFO] Evaluating models...")
    summary_rows = []
    prediction_rows = []
    final_param_rows = []

    for name, theta, fit_time, hist in [
        ("Gradient Descent", gd_theta, gd_fit_time, gd_history),
        ("Gauss-Newton", gn_theta, gn_fit_time, gn_history),
        ("L-BFGS", lbfgs_theta, lbfgs_fit_time, lbfgs_history),
    ]:
        metrics, preds = evaluate_model(
            name,
            theta,
            X_train_s, y_train_s,
            X_val_s, y_val_s,
            X_test_s, y_test_s,
            d, h, y_scaler,
        )
        metrics["n_iterations"] = int(hist["iteration"].max())
        metrics["iteration_budget"] = int(COMMON_MAX_ITER)
        metrics["fit_time_sec"] = float(fit_time)
        summary_rows.append(metrics)
        prediction_rows.append(preds)

        for idx, value in enumerate(theta):
            final_param_rows.append({
                "optimizer": name,
                "parameter_index": idx,
                "parameter_value": float(value),
            })

    summary_df = pd.DataFrame(summary_rows).sort_values("test_rmse", ascending=True).reset_index(drop=True)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)
    final_params_df = pd.DataFrame(final_param_rows)

    history_df.to_csv(WORKING_DIR / "optimizer_history.csv", index=False)
    summary_df.to_csv(WORKING_DIR / "optimizer_summary.csv", index=False)
    predictions_df.to_csv(WORKING_DIR / "all_predictions_and_residuals.csv", index=False)
    final_params_df.to_csv(WORKING_DIR / "final_parameters.csv", index=False)

    preprocessing_info = pd.DataFrame([
        {
            "dataset_path": str(dataset_path),
            "n_rows": int(df.shape[0]),
            "n_features": int(len(feature_cols)),
            "target_col": target_col,
            "random_state": RANDOM_STATE,
            "hidden_dim": HIDDEN_DIM,
            "regularization": REG,
            "gd_learning_rate": GD_LR,
            "common_max_iter": COMMON_MAX_ITER,
            "use_early_stopping": USE_EARLY_STOPPING,
            "gn_damping": GN_DAMPING,
        }
    ])
    preprocessing_info.to_csv(WORKING_DIR / "experiment_config.csv", index=False)

    # JSON too, for quick reuse
    (WORKING_DIR / "experiment_config.json").write_text(
        json.dumps(preprocessing_info.iloc[0].to_dict(), indent=2), encoding="utf-8"
    )

    print("[INFO] Saving plots...")
    save_plot_loss_curves(history_df)
    save_plot_metrics(summary_df)
    save_prediction_and_residual_plots(predictions_df)

    dataset_meta = {
        "n_rows": int(df.shape[0]),
        "n_features": int(len(feature_cols)),
        "target_col": target_col,
    }
    create_section4_markdown(summary_df, history_df, dataset_meta)

    print("\n===== FINAL SUMMARY =====")
    print(summary_df.to_string(index=False))
    print("\nOutput files saved to:")
    print(f"- {WORKING_DIR / 'optimizer_history.csv'}")
    print(f"- {WORKING_DIR / 'optimizer_summary.csv'}")
    print(f"- {WORKING_DIR / 'all_predictions_and_residuals.csv'}")
    print(f"- {WORKING_DIR / 'final_parameters.csv'}")
    print(f"- {WORKING_DIR / 'experiment_config.csv'}")
    print(f"- {WORKING_DIR / 'section_4_hasil_dan_pembahasan.md'}")
    print(f"- {PLOTS_DIR}")


if __name__ == "__main__":
    main()


[INFO] Loading dataset...
[INFO] Fair comparison mode: same initialization, same data split, same objective, same iteration budget = 1000
[INFO] Training Gradient Descent...
[INFO] Training Gauss-Newton...
[INFO] Training L-BFGS...
[INFO] Evaluating models...
[INFO] Saving plots...

===== FINAL SUMMARY =====
       optimizer  train_rmse  val_rmse  test_rmse  train_mae  val_mae  test_mae  train_r2   val_r2  test_r2  test_mape  n_iterations  iteration_budget  fit_time_sec
    Gauss-Newton    5.134084  6.461132   5.854011   3.890975 4.809439  4.583371  0.901825 0.875029 0.869123  14.018718          1000              1000      1.412172
          L-BFGS    4.833338  6.540896   5.864950   3.736914 4.849807  4.523154  0.912990 0.871924 0.868633  14.944879          1000              1000      0.762407
Gradient Descent    6.933664  8.028818   6.282098   5.362120 5.950047  4.914409  0.820938 0.807027 0.849281  17.339799          1000              1000      0.251736

Output files saved to:
- /kag